# Combine per-experiment `analysis.csv` files into a single combined frame

Concatenates the per-experiment `analysis.csv` files (one per imaging experiment) into a single combined dataframe and computes derived per-row metrics (`cell area`, `% compaction`, integrated densities).

Two modes, controlled by the config cell:

1. **First-time combine.** No prior combined frame exists. The notebook concatenates the per-experiment inputs, computes derived columns, and writes a fresh combined CSV.
2. **Update with curated annotations.** A prior combined CSV exists and contains curated per-row columns (e.g. `omit`, `actin omit`, `actin omit reason`) that should be carried forward. The notebook left-merges those columns onto the freshly concatenated frame on the row keys, validates that the curated values round-trip exactly, then writes a new combined CSV.

The notebook never overwrites an existing output: results are written to a date-versioned filename, and a JSON manifest with input SHA-256 hashes, the git commit, and a UTC timestamp is saved alongside each output.

In [ ]:
from pathlib import Path
import datetime
import hashlib
import json
import subprocess

import numpy as np
import pandas as pd

## Config

All paths and parameters for a run are declared here. Nothing below this cell needs editing.

- `INPUTS`: per-experiment `analysis.csv` files keyed by experiment ID.
- `COMBINED_PATH`: existing combined CSV whose curated columns should be carried forward. Set to `None` for a first-time combine.
- `CURATED_COLS`: column names in the existing combined CSV to preserve through the merge. Ignored when `COMBINED_PATH is None`.
- `MERGE_KEYS`: columns that uniquely identify a row across all inputs. Used as the join key and validated for uniqueness.
- `COLUMN_RENAMES`: optional mapping applied to every per-experiment frame before concatenation, used to harmonize column names when the same quantity is recorded under different headers across experiments. Renames whose source column is absent are silently ignored.
- `OUTPUT_DIR`: directory for the combined CSV and its manifest.

In [ ]:
INPUTS = {
    'CE029': Path('/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab_Box/Z-lab shared folders/Kathryn + Eduardo/CE029/caax488_memglow647/img_processing/tables/analysis.csv'),
    'CE030': Path('/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab_Box/Z-lab shared folders/Kathryn + Eduardo/CE030_1st data set_JLY-LatA-Jasp-treatment/img_processing/tables/analysis.csv'),
    'CE031': Path('/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab_Box/Z-lab shared folders/Kathryn + Eduardo/CE031/experiment/img_processing/tables/analysis.csv'),
}

# Set to None for a first-time combine; otherwise point at the existing combined CSV.
COMBINED_PATH = Path('/Users/kwu2/Library/CloudStorage/GoogleDrive-kwu2@stanford.edu/My Drive/Lab/OL_compaction/Experiments_compiled/actin_pharmacological/data/combined_analysis.csv')

# Curated columns from COMBINED_PATH to preserve through the merge.
CURATED_COLS = ['omit', 'actin omit', 'actin omit reason']

# Row identity across all inputs.
MERGE_KEYS = ['UID', 't']

# Column-name harmonization applied to each per-experiment frame before concat.
# Add entries here when the same measurement is recorded under different names across experiments.
COLUMN_RENAMES: dict[str, str] = {}

OUTPUT_DIR = Path('/Users/kwu2/Library/CloudStorage/GoogleDrive-kwu2@stanford.edu/My Drive/Lab/OL_compaction/Experiments_compiled/actin_pharmacological/data')

TODAY = datetime.date.today().isoformat()
OUTPUT_PATH = OUTPUT_DIR / f'combined_analysis_{TODAY}.csv'
MANIFEST_PATH = OUTPUT_DIR / f'combined_analysis_{TODAY}.manifest.json'

## Load inputs

Per-experiment frames are loaded into a dict keyed by experiment ID. The existing combined frame is loaded only when `COMBINED_PATH` is provided and the file exists; otherwise the notebook runs in first-time-combine mode.

In [ ]:
per_exp = {k: pd.read_csv(p) for k, p in INPUTS.items()}
for k, df in per_exp.items():
    print(f'{k}: {df.shape[0]} rows, {df.shape[1]} cols')

if COMBINED_PATH is not None and Path(COMBINED_PATH).exists():
    combined = pd.read_csv(COMBINED_PATH)
    print(f'combined: {combined.shape[0]} rows, {combined.shape[1]} cols')
    update_mode = True
else:
    combined = None
    update_mode = False
    print('No existing combined CSV provided; running first-time combine.')

## Harmonize column names

Applies `COLUMN_RENAMES` to every per-experiment frame so that columns measuring the same quantity share a single canonical name before concatenation. This is the single point at which schema variation across experiments is reconciled.

In [ ]:
if COLUMN_RENAMES:
    per_exp = {k: df.rename(columns=COLUMN_RENAMES) for k, df in per_exp.items()}
    print(f'Applied {len(COLUMN_RENAMES)} canonical rename(s).')
else:
    print('No canonical renames configured.')

## Validate row identity

Each per-experiment frame, and the existing combined frame when present, must have unique values of `MERGE_KEYS`. A duplicate key would cause the merge to fan out, inflating the row count of the output.

In [ ]:
for k, df in per_exp.items():
    dups = df.duplicated(MERGE_KEYS).sum()
    assert dups == 0, f'{k}: {dups} duplicate {MERGE_KEYS} rows'

if update_mode:
    dups = combined.duplicated(MERGE_KEYS).sum()
    assert dups == 0, f'combined: {dups} duplicate {MERGE_KEYS} rows'

print(f'All inputs have unique {MERGE_KEYS}.')

## Inspect schema alignment across per-experiment files

`pd.concat` aligns columns by exact header match; columns absent from a given experiment become NaN for that experiment's rows. The block below lists any column that is not present in every per-experiment input, so genuine biological asymmetries (e.g. an experiment in which a particular event did not occur) can be distinguished from header inconsistencies that should be reconciled via `COLUMN_RENAMES`.

In [ ]:
all_cols = {k: set(df.columns) for k, df in per_exp.items()}
shared = set.intersection(*all_cols.values())
union = set.union(*all_cols.values())
asymmetric = union - shared

if asymmetric:
    print(f'Columns NOT shared across all per-experiment inputs ({len(asymmetric)}):')
    for c in sorted(asymmetric):
        present_in = [k for k, cols in all_cols.items() if c in cols]
        print(f'  {c} -> present in {present_in}')
else:
    print('All per-experiment files have identical column sets.')

## Concatenate per-experiment frames

In [ ]:
concat = pd.concat(list(per_exp.values()), ignore_index=True)
print(f'Concatenated shape: {concat.shape}')
assert concat.shape[0] == sum(df.shape[0] for df in per_exp.values()), \
    'row count mismatch after concat'

## Carry curated columns forward (update mode only)

When an existing combined CSV is provided, its curated columns are joined onto the concatenated frame on `MERGE_KEYS`. The `validate='one_to_one'` argument guarantees that each row of the combined frame matches exactly one row of the concatenated frame; a non-1:1 relationship raises immediately. In first-time-combine mode this cell is a no-op.

In [ ]:
if update_mode:
    missing = [c for c in CURATED_COLS if c not in combined.columns]
    assert not missing, f'combined missing curated columns: {missing}'
    curated = combined[MERGE_KEYS + CURATED_COLS]
    merged = concat.merge(curated, on=MERGE_KEYS, how='left', validate='one_to_one')
    print(f'Merged shape: {merged.shape}')
else:
    merged = concat
    print('First-time combine: no curated columns to merge.')

## Validate merge integrity (update mode only)

Two checks: (i) row count is unchanged by the merge, and (ii) every curated value in the original combined frame is reproduced exactly in the merged frame, compared NaN-for-NaN. If either fails, the saved output should be discarded and the inputs reconciled.

In [ ]:
if update_mode:
    assert merged.shape[0] == concat.shape[0], \
        f'row count changed during merge: {concat.shape[0]} -> {merged.shape[0]}'

    if merged.shape[0] != combined.shape[0]:
        diff = merged.shape[0] - combined.shape[0]
        print(f'WARNING: merged has {diff:+d} rows vs. combined ({merged.shape[0]} vs {combined.shape[0]}).')
        print('Investigate before treating this output as canonical.')
    else:
        print('Row counts match between merged and combined.')

    m_check = merged.set_index(MERGE_KEYS)[CURATED_COLS].sort_index()
    c_check = combined.set_index(MERGE_KEYS)[CURATED_COLS].sort_index()
    common_idx = m_check.index.intersection(c_check.index)
    assert m_check.loc[common_idx].equals(c_check.loc[common_idx]), \
        'curated columns differ between merged output and original combined'
    print(f'Curated column integrity verified across {len(common_idx)} keys.')
else:
    print('First-time combine: no merge integrity check required.')

## Compute derived per-row metrics

`cell area` is the union of compacted and CAAX-positive areas. `% compaction` is the fraction of cell area that is compacted. Integrated density columns are mean intensity over the cell scaled by cell area, giving a quantity proportional to total signal.

In [ ]:
merged['cell area'] = merged['CAAX-positive area'] + merged['compacted area']
merged['% compaction'] = merged['compacted area'] / merged['cell area']
merged['caax integrated density'] = merged['mean caax int (cell)'] * merged['cell area']
merged['actin integrated density'] = merged['mean actin int (cell)'] * merged['cell area']

print(f'Final shape: {merged.shape}')
merged.head()

## Save output and provenance manifest

The combined CSV is written to a date-versioned filename and the run refuses to overwrite an existing file. A JSON manifest written alongside records the SHA-256 hash and row/column count of every input, the SHA-256 of the output, the git commit of the analysis repository, and a UTC timestamp, providing a self-contained record of how this output was produced.

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def git_commit(repo_dir):
    try:
        return subprocess.check_output(
            ['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert not OUTPUT_PATH.exists(), f'refusing to overwrite existing file: {OUTPUT_PATH}'
merged.to_csv(OUTPUT_PATH, index=False)

manifest = {
    'mode': 'update' if update_mode else 'first_time_combine',
    'output': str(OUTPUT_PATH),
    'output_sha256': file_sha256(OUTPUT_PATH),
    'output_rows': int(merged.shape[0]),
    'output_cols': int(merged.shape[1]),
    'inputs': {
        k: {'path': str(p), 'sha256': file_sha256(p)} for k, p in INPUTS.items()
    },
    'combined_input': (
        {'path': str(COMBINED_PATH), 'sha256': file_sha256(COMBINED_PATH)}
        if update_mode else None
    ),
    'merge_keys': MERGE_KEYS,
    'curated_cols_preserved': CURATED_COLS if update_mode else [],
    'column_renames_applied': COLUMN_RENAMES,
    'git_commit': git_commit(Path.cwd()),
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
with open(MANIFEST_PATH, 'w') as fh:
    json.dump(manifest, fh, indent=2)

print(f'Wrote {OUTPUT_PATH}')
print(f'Wrote {MANIFEST_PATH}')